# 01_curation_true_positive_set


In [11]:
from rdkit import Chem
from rdkit.Chem import rdMolDescriptors
import pandas as pd

print("Todo importado correctamente ✅")

Todo importado correctamente ✅


In [12]:
%pip install requests

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [13]:
compounds = [
    # (Compound_ID, Name, Formula_Reported, Extract, Class)
    ("B01", "Dihydroferulic acid 4-sulfate", "C10H12O7S", "EMB", "PC"),
    ("B02", "Quinic acid", "C7H12O6", "EMB;EVB", "PC"),
    ("B03", "Pyrocatechol", "C6H6O2", "EVB", "PC"),
    ("B04", "Chlorogenic acid", "C16H18O9", "EMB;EVB", "PC"),
    ("B05", "4-Hydroxybenzoic acid", "C7H6O3", "EMB;EVB", "PC"),
    ("B06", "p-Coumaroylquinic acid", "C16H18O8", "EMB;EVB", "PC"),
    ("B09", "Vanillin", "C8H8O3", "EVB", "PC"),
    ("B10", "Feruloylquinic acid", "C17H20O9", "EMB;EVB", "PC"),
    ("B11", "Vanillic acid", "C8H8O4", "EVB", "PC"),
    ("B12", "Caffeic acid", "C9H8O4", "EMB;EVB", "PC"),
    ("B13", "4-Hydroxybenzaldehyde", "C7H6O2", "EMB;EVB", "PC"),
    ("B14", "2,4-Dihydroxybenzoic acid", "C7H6O4", "EMB;EVB", "PC"),
    ("B17", "p-Coumaric acid", "C9H8O3", "EMB;EVB", "PC"),
    ("B18", "Schaftoside", "C26H28O14", "EMB;EVB", "PC"),
    ("B19", "Vicenin-2", "C27H30O15", "EMB", "PC"),
    ("B20", "3,4-Dihydroxybenzaldehyde", "C7H6O3", "EMB;EVB", "PC"),
    ("B21", "Methoxyluteolin-8-C-glucoside", "C22H22O11", "EMB;EVB", "PC"),
    ("B22", "Ferulic acid", "C10H10O4", "EVB", "PC"),
    ("A03", "4-Hydroxy-2,5-dimethyl-3(2H)-furanone", "C6H8O3", "EVA", "HIC"),
    ("A05", "3-Methylcyclopentane-1,2,4-trione", "C6H6O3", "EMA", "HIC"),
    ("A06", "Glycerol", "C3H8O3", "EMA", "HIC"),
    ("A07", "2,3-Dihydro-3,5-dihydroxy-6-methyl-4H-pyran-4-one", "C6H8O4", "EMA;EVA", "HIC"),
    ("A09", "2-Methoxy-4-vinylphenol", "C9H10O2", "EVA", "HIC"),
    ("A10", "4-Methylcatechol", "C7H8O2", "EMA", "HIC"),
    ("A13", "2,6-Dimethoxyphenol", "C8H10O3", "EVA", "HIC"),
    ("A15", "Pyrogallol", "C6H6O3", "EMA", "HIC"),
    ("A17", "2,4-Di-tert-butylphenol", "C14H22O", "EVA", "HIC"),
    ("A18", "Levoglucosan", "C6H10O5", "EVA", "HIC"),
    ("A19", "4-Butyl-2-methoxyphenol", "C11H16O2", "EVA", "HIC"),
    ("A22", "1-Hydroxycyclohexyl phenyl ketone", "C13H16O2", "EMA;EVA", "HIC"),
    ("A23", "Myristic acid", "C14H28O2", "EMA", "HIC"),
    ("A24", "Palmitic acid", "C16H32O2", "EMA;EVA", "HIC"),
    ("A25", "3-(3,5-Di-tert-butyl-4-hydroxyphenyl)propionic acid", "C17H26O3", "EMA;EVA", "HIC"),
    ("A26", "Vaccenic acid", "C18H34O2", "EVA", "HIC"),
    ("A27", "Stearic acid", "C18H36O2", "EMA", "HIC"),
    ("A28", "Arachidic acid", "C20H40O2", "EMA", "HIC"),
    ("A29", "3,4-Dimethoxyphenol", "C8H10O3", "EVA", "HIC"),
]

print(f"Total de compuestos en la lista: {len(compounds)}")

Total de compuestos en la lista: 37


## 📋 Registro de decisiones de curación (Compound List)

Comparado con las Tablas 3 y 4 originales del paper, se hicieron los siguientes ajustes:

### Renombres (para mejorar el match con nombres indexados en PubChem)
| ID | Nombre original (paper) | Nombre usado aquí | Razón |
|---|---|---|---|
| B19 | Apigenin-6,8-C-diglucoside | Vicenin-2 | Sinónimo más común en bases de datos |
| A18 | 1,6-Anhydro-β-D-glucopyranose | Levoglucosan | Sinónimo más común en bases de datos |

### Unificación de isómeros (mismo nombre base, distinta posición de acilación no resuelta en el paper)
| IDs originales fusionados | Nombre unificado | Fórmula |
|---|---|---|
| B04, B07, B08 | Chlorogenic acid | C16H18O9 |
| B06, B15 | p-Coumaroylquinic acid | C16H18O8 |
| B10, B16 | Feruloylquinic acid | C17H20O9 |

*Nota: se pierden los datos de abundancia relativa por isómero individual al fusionar;
si se necesita ese nivel de detalle más adelante, referirse a la Tabla 3 original en
`data/raw/table3_PCs.csv`.*

### Exclusiones (probables artefactos de GC, no metabolitos genuinos)
`A01, A02, A04, A08, A11, A12, A14, A16, A20, A21, A30`
— alcanos/dioxolanos/ésteres de fosfato típicos de contaminación de columna o plastificantes.

### Total final
**37 compuestos únicos** (18 PC + 19 HIC), tras renombres, fusión de isómeros y exclusión
de artefactos.

In [14]:
import requests

# Probamos con un compuesto que ya conocemos bien: caffeic acid
nombre_prueba = "caffeic acid"
url_prueba = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/{nombre_prueba}/property/CanonicalSMILES,InChIKey,MolecularFormula/JSON"

respuesta = requests.get(url_prueba, timeout=10)
print(f"Código de estado HTTP: {respuesta.status_code}")
print(respuesta.json())

Código de estado HTTP: 200
{'PropertyTable': {'Properties': [{'CID': 689043, 'MolecularFormula': 'C9H8O4', 'ConnectivitySMILES': 'C1=CC(=C(C=C1C=CC(=O)O)O)O', 'InChIKey': 'QAIPRVGONGVQAS-DUXPYHPUSA-N'}]}}


## 🌐 Consulta automática a PubChem vía API (PUG REST)

En vez de buscar cada compuesto manualmente en el navegador (como se hizo inicialmente
con B01, a modo de prueba), esta sección consulta la API pública de PubChem de forma
programática

PUG REST (*Power User Gateway - REST*) es la interfaz pública que PubChem ofrece para
consultar su base de datos mediante peticiones HTTP simples — la misma tecnología que
usa cualquier página web para pedir datos a un servidor, pero aquí el "servidor" es la
base de datos química de PubChem y la "respuesta" es un archivo JSON estructurado en
vez de una página visual.

### Estructura de la URL de consulta

| Parte de la URL | Qué significa |
|---|---|
| `compound/name/{NOMBRE}` | Busca un compuesto por su nombre textual (ej. "caffeic acid") |
| `property/{PROPIEDADES}` | Lista de propiedades específicas a devolver, separadas por comas |
| `JSON` | Formato de salida solicitado (también podría pedirse en XML, CSV, etc.) |

En este proyecto se solicitan tres propiedades: `ConnectivitySMILES`, `InChIKey`, y
`MolecularFormula`.

> ⚠️ **Nota de depuración real:** inicialmente se intentó pedir la propiedad
> `CanonicalSMILES` (nombre usado en versiones anteriores de la documentación de
> PubChem), pero la API actual devuelve esa información bajo la clave
> `ConnectivitySMILES`. Se detectó al inspeccionar la respuesta real de una
> consulta de prueba antes de automatizar el proceso completo — de ahí la
> importancia de probar con un solo caso antes de correr un loop masivo.

### Estructura de la respuesta JSON

PubChem devuelve un diccionario anidado con esta forma:

```json
{
    "PropertyTable": {
        "Properties": [
            {
                "CID": 689043,
                "MolecularFormula": "C9H8O4",
                "ConnectivitySMILES": "C1=CC(=C(C=C1C=CC(=O)O)O)O",
                "InChIKey": "QAIPRVGONGVQAS-DUXPYHPUSA-N"
            }
        ]
    }
}
```

- `Properties` es siempre una **lista**, aunque solo devuelva un resultado — por eso
  el código accede al primer elemento con `[0]`.
- `CID` (*Compound ID*) es el identificador numérico único de PubChem para esa
  molécula — útil como referencia cruzada en pasos posteriores del proyecto.

### Manejo de errores

La función `buscar_pubchem()` incluye un bloque `try/except` porque una consulta a un
servidor externo puede fallar por múltiples razones ajenas al código (compuesto no
encontrado, tiempo de espera agotado, conexión interrumpida). Si el compuesto no se
encuentra, se conserva su registro en la tabla final con los campos de PubChem vacíos,
en lugar de perderlo silenciosamente — permitiendo su revisión manual posterior.

Adicionalmente, se incluye una pausa (`time.sleep(0.3)`) entre cada consulta como
cortesía técnica hacia el servidor público de PubChem, evitando que un volumen alto
de peticiones seguidas sea interpretado como tráfico abusivo.

In [15]:
import time

def buscar_pubchem(nombre_compuesto):
    """
    Busca un compuesto por nombre en PubChem (PUG REST API) y devuelve
    su SMILES, InChIKey, fórmula molecular y CID.

    Parameters
    ----------
    nombre_compuesto : str
        Nombre del compuesto a buscar (en inglés).

    Returns
    -------
    dict o None
        Diccionario con las claves: cid, smiles, inchikey, formula_pubchem.
        Devuelve None si no se encontró el compuesto o hubo un error.
    """
    nombre_codificado = requests.utils.quote(nombre_compuesto)
    url = (
        f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/"
        f"{nombre_codificado}/property/"
        f"ConnectivitySMILES,InChIKey,MolecularFormula/JSON"
    )

    try:
        respuesta = requests.get(url, timeout=10)

        if respuesta.status_code != 200:
            print(f"  ⚠️ '{nombre_compuesto}': código HTTP {respuesta.status_code} (no encontrado)")
            return None

        datos = respuesta.json()
        propiedades = datos["PropertyTable"]["Properties"][0]

        return {
            "cid": propiedades.get("CID"),
            "smiles": propiedades.get("ConnectivitySMILES"),
            "inchikey": propiedades.get("InChIKey"),
            "formula_pubchem": propiedades.get("MolecularFormula"),
        }

    except Exception as e:
        print(f"  ⚠️ Error buscando '{nombre_compuesto}': {e}")
        return None


# Prueba rápida de la función con el mismo compuesto de antes
resultado_prueba = buscar_pubchem("caffeic acid")
print(resultado_prueba)

{'cid': 689043, 'smiles': 'C1=CC(=C(C=C1C=CC(=O)O)O)O', 'inchikey': 'QAIPRVGONGVQAS-DUXPYHPUSA-N', 'formula_pubchem': 'C9H8O4'}


In [16]:
resultados_pubchem = []

for compound_id, name, formula_reportada, extract, clase in compounds:
    print(f"Buscando: {compound_id} - {name}...")

    datos = buscar_pubchem(name)

    if datos is None:
        # Guardamos igual el registro, pero con los campos de PubChem vacíos,
        # para no perder el compuesto de la lista — lo resolveremos manualmente después
        resultados_pubchem.append({
            "Compound_ID": compound_id,
            "Name": name,
            "SMILES": None,
            "InChIKey": None,
            "Formula_Reported": formula_reportada,
            "Formula_PubChem": None,
            "Extract": extract,
            "Class": clase,
        })
    else:
        resultados_pubchem.append({
            "Compound_ID": compound_id,
            "Name": name,
            "SMILES": datos["smiles"],
            "InChIKey": datos["inchikey"],
            "Formula_Reported": formula_reportada,
            "Formula_PubChem": datos["formula_pubchem"],
            "Extract": extract,
            "Class": clase,
        })

    time.sleep(0.3)  # pausa breve para no saturar el servidor de PubChem

print(f"\n✅ Proceso terminado. {len(resultados_pubchem)} compuestos procesados.")

Buscando: B01 - Dihydroferulic acid 4-sulfate...
Buscando: B02 - Quinic acid...
Buscando: B03 - Pyrocatechol...
Buscando: B04 - Chlorogenic acid...
Buscando: B05 - 4-Hydroxybenzoic acid...
Buscando: B06 - p-Coumaroylquinic acid...
Buscando: B09 - Vanillin...
Buscando: B10 - Feruloylquinic acid...
  ⚠️ 'Feruloylquinic acid': código HTTP 404 (no encontrado)
Buscando: B11 - Vanillic acid...
Buscando: B12 - Caffeic acid...
Buscando: B13 - 4-Hydroxybenzaldehyde...
Buscando: B14 - 2,4-Dihydroxybenzoic acid...
Buscando: B17 - p-Coumaric acid...
Buscando: B18 - Schaftoside...
Buscando: B19 - Vicenin-2...
Buscando: B20 - 3,4-Dihydroxybenzaldehyde...
Buscando: B21 - Methoxyluteolin-8-C-glucoside...
  ⚠️ 'Methoxyluteolin-8-C-glucoside': código HTTP 404 (no encontrado)
Buscando: B22 - Ferulic acid...
Buscando: A03 - 4-Hydroxy-2,5-dimethyl-3(2H)-furanone...
Buscando: A05 - 3-Methylcyclopentane-1,2,4-trione...
Buscando: A06 - Glycerol...
Buscando: A07 - 2,3-Dihydro-3,5-dihydroxy-6-methyl-4H-pyran-4-

In [17]:
# Resultados encontrados manualmente en PubChem (los 2 casos que la API no matcheó por nombre)
hallazgos_manuales = {
    "B10": {
        "smiles": "COC1=C(C=CC(=C1)C=CC(=O)O[C@@H]2C[C@](C[C@H]([C@@H]2O)O)(C(=O)O)O)O",
        "inchikey": "RAGZUCNPTLULOL-JSHWQEIDSA-N",
        "formula_pubchem": "C17H20O9",
        "fuente": "Búsqueda manual en PubChem (nombre indexado: '5-O-Feruloylquinic acid')"
    },
    "B21": {
        "smiles": "COC1=C(C=CC(=C1)C2=CC(=O)C3=C(O2)C(=C(C=C3O)O)[C@H]4[C@@H]([C@H]([C@@H]([C@H](O4)CO)O)O)O)O",
        "inchikey": "YXHFXGHAELQJGK-PGPONNFDSA-N",
        "formula_pubchem": "C22H22O11",
        "fuente": "Búsqueda manual en PubChem (nombre indexado: 'Scoparin')"
    },
}

for resultado in resultados_pubchem:
    if resultado["Compound_ID"] in hallazgos_manuales:
        datos_manuales = hallazgos_manuales[resultado["Compound_ID"]]
        resultado["SMILES"] = datos_manuales["smiles"]
        resultado["InChIKey"] = datos_manuales["inchikey"]
        resultado["Formula_PubChem"] = datos_manuales["formula_pubchem"]
        resultado["Fuente"] = datos_manuales["fuente"]
        print(f"✅ {resultado['Compound_ID']} actualizado manualmente")

print("\nListo — hallazgos manuales incorporados a resultados_pubchem")

✅ B10 actualizado manualmente
✅ B21 actualizado manualmente

Listo — hallazgos manuales incorporados a resultados_pubchem


In [18]:
import sys
sys.path.append('..')  # le dice a Python que también busque módulos un nivel arriba

from src.validation import validar_compuesto

# Prueba rápida con un compuesto conocido
formula_prueba, inchikey_prueba, smiles_prueba = validar_compuesto("OC(=O)c1ccc(O)cc1")
print(f"Fórmula:  {formula_prueba}")
print(f"InChIKey: {inchikey_prueba}")
print(f"SMILES:   {smiles_prueba}")

Fórmula:  C7H6O3
InChIKey: FJKROLUGYXJWQN-UHFFFAOYSA-N
SMILES:   O=C(O)c1ccc(O)cc1


In [20]:
for resultado in resultados_pubchem:
    smiles = resultado["SMILES"]

    if smiles is None:
        # No debería pasar ya (los 2 casos especiales se resolvieron manualmente),
        # pero lo dejamos como salvaguarda
        resultado["Formula_RDKit"] = None
        resultado["InChIKey_RDKit"] = None
        resultado["SMILES_Canonical"] = None
        resultado["Match_vs_Paper"] = False
        continue

    formula_rdkit, inchikey_rdkit, smiles_canonico = validar_compuesto(smiles)

    resultado["Formula_RDKit"] = formula_rdkit
    resultado["InChIKey_RDKit"] = inchikey_rdkit
    resultado["SMILES_Canonical"] = smiles_canonico
    resultado["Match_vs_Paper"] = (formula_rdkit == resultado["Formula_Reported"])

# Resumen general
total = len(resultados_pubchem)
coincidencias = sum(1 for r in resultados_pubchem if r["Match_vs_Paper"])

print(f"Total de compuestos:                {total}")
print(f"Fórmula coincide con el paper:      {coincidencias}")
print(f"Fórmula NO coincide (revisar):      {total - coincidencias}")

if coincidencias < total:
    print("\n⚠️ Compuestos con fórmula que no coincide:")
    for r in resultados_pubchem:
        if not r["Match_vs_Paper"]:
            print(f"  {r['Compound_ID']} - {r['Name']}: "
                  f"paper={r['Formula_Reported']} vs RDKit={r['Formula_RDKit']}")

Total de compuestos:                37
Fórmula coincide con el paper:      37
Fórmula NO coincide (revisar):      0


[13:57:24] WARNING: Omitted undefined stereo

[13:57:24] WARNING: Omitted undefined stereo

[13:57:24] WARNING: Omitted undefined stereo

[13:57:24] WARNING: Omitted undefined stereo

[13:57:24] WARNING: Omitted undefined stereo

[13:57:24] WARNING: Omitted undefined stereo

[13:57:24] WARNING: Omitted undefined stereo

[13:57:24] WARNING: Omitted undefined stereo

[13:57:24] WARNING: Omitted undefined stereo

[13:57:24] WARNING: Omitted undefined stereo

[13:57:24] WARNING: Omitted undefined stereo

[13:57:24] WARNING: Omitted undefined stereo

[13:57:24] WARNING: Omitted undefined stereo

[13:57:24] WARNING: Omitted undefined stereo



In [21]:
df_final = pd.DataFrame(resultados_pubchem)

# Reordenamos las columnas para que coincidan con el formato pedido por la guía:
# Compound_ID | Name | SMILES | InChIKey | Formula | Extract | Relative_Abundance | Class
columnas_finales = [
    "Compound_ID", "Name", "SMILES_Canonical", "InChIKey_RDKit",
    "Formula_Reported", "Extract", "Class",
    "Formula_PubChem", "Formula_RDKit", "Match_vs_Paper"
]

df_final = df_final[columnas_finales]

# Renombramos para que coincida exactamente con el nombre pedido en la guía
df_final = df_final.rename(columns={
    "SMILES_Canonical": "SMILES",
    "InChIKey_RDKit": "InChIKey",
    "Formula_Reported": "Formula",
})

print(f"Dimensiones de la tabla: {df_final.shape[0]} filas x {df_final.shape[1]} columnas")
df_final

Dimensiones de la tabla: 37 filas x 10 columnas


,Compound_ID,Name,SMILES,InChIKey,Formula,Extract,Class,Formula_PubChem,Formula_RDKit,Match_vs_Paper
0,B01,Dihydroferulic acid 4-sulfate,COc1cc(CCC(=O)O)ccc1OS(=O)(=O)O,UMCDODPBPQMWQP-UHFFFAOYSA-N,C10H12O7S,EMB,PC,C10H12O7S,C10H12O7S,True
1,B02,Quinic acid,O=C(O)C1(O)CC(O)C(O)C(O)C1,AAWZDTNXLSGCEK-UHFFFAOYSA-N,C7H12O6,EMB;EVB,PC,C7H12O6,C7H12O6,True
2,B03,Pyrocatechol,Oc1ccccc1O,YCIMNLLNPGFGHC-UHFFFAOYSA-N,C6H6O2,EVB,PC,C6H6O2,C6H6O2,True
3,B04,Chlorogenic acid,O=C(C=Cc1ccc(O)c(O)c1)OC1CC(O)(C(=O)O)CC(O)C1O,CWVRJTMFETXNAD-UHFFFAOYSA-N,C16H18O9,EMB;EVB,PC,C16H18O9,C16H18O9,True
4,B05,4-Hydroxybenzoic acid,O=C(O)c1ccc(O)cc1,FJKROLUGYXJWQN-UHFFFAOYSA-N,C7H6O3,EMB;EVB,PC,C7H6O3,C7H6O3,True
5,B06,p-Coumaroylquinic acid,O=C(C=Cc1ccc(O)cc1)OC1CC(O)(C(=O)O)CC(O)C1O,BMRSEYFENKXDIS-UHFFFAOYSA-N,C16H18O8,EMB;EVB,PC,C16H18O8,C16H18O8,True
6,B09,Vanillin,COc1cc(C=O)ccc1O,MWOOGOJBHIARFG-UHFFFAOYSA-N,C8H8O3,EVB,PC,C8H8O3,C8H8O3,True
7,B10,Feruloylquinic acid,COc1cc(C=CC(=O)O[C@@H]2C[C@@](O)(C(=O)O)C[C@@H...,RAGZUCNPTLULOL-JSHWQEIDSA-N,C17H20O9,EMB;EVB,PC,C17H20O9,C17H20O9,True
8,B11,Vanillic acid,COc1cc(C(=O)O)ccc1O,WKOLLVMJNQIZCI-UHFFFAOYSA-N,C8H8O4,EVB,PC,C8H8O4,C8H8O4,True
9,B12,Caffeic acid,O=C(O)C=Cc1ccc(O)c(O)c1,QAIPRVGONGVQAS-UHFFFAOYSA-N,C9H8O4,EMB;EVB,PC,C9H8O4,C9H8O4,True


In [22]:
ruta_salida = "../data/processed/step1_true_positive_compounds.csv"
df_final.to_csv(ruta_salida, index=False)

print(f"✅ Tabla guardada en: {ruta_salida}\n")

print("="*60)
print("CHECKLIST DEL ENTREGABLE 1")
print("="*60)
print(f"[{'✅' if len(df_final) >= 30 else '❌'}] Mínimo 30-35 compuestos → tienes {len(df_final)}")
print(f"[{'✅' if df_final['Compound_ID'].is_unique else '❌'}] Compound_ID sin duplicados")
print(f"[{'✅' if df_final['SMILES'].notna().all() else '❌'}] Todos los compuestos tienen SMILES")
print(f"[{'✅' if df_final['InChIKey'].notna().all() else '❌'}] Todos los compuestos tienen InChIKey")
print(f"[{'✅' if df_final['Match_vs_Paper'].all() else '⚠️'}] Todas las fórmulas coinciden con el paper "
      f"({df_final['Match_vs_Paper'].sum()}/{len(df_final)})")

columnas_requeridas = {"Compound_ID", "Name", "SMILES", "InChIKey", "Formula", "Extract", "Class"}
columnas_presentes = set(df_final.columns)
faltantes = columnas_requeridas - columnas_presentes
print(f"[{'✅' if not faltantes else '❌'}] Columnas mínimas requeridas presentes"
      f"{' (faltan: ' + str(faltantes) + ')' if faltantes else ''}")

✅ Tabla guardada en: ../data/processed/step1_true_positive_compounds.csv

CHECKLIST DEL ENTREGABLE 1
[✅] Mínimo 30-35 compuestos → tienes 37
[✅] Compound_ID sin duplicados
[✅] Todos los compuestos tienen SMILES
[✅] Todos los compuestos tienen InChIKey
[✅] Todas las fórmulas coinciden con el paper (37/37)
[✅] Columnas mínimas requeridas presentes


In [23]:
tabla3_original = pd.read_csv("../data/raw/table3_PCs.csv")
tabla4_original = pd.read_csv("../data/raw/table4_HICs.csv")

print("Tabla 3 (PCs):")
print(tabla3_original.head())
print(f"\nTabla 4 (HICs):")
print(tabla4_original.head())

Tabla 3 (PCs):
  Compound_ID  Retention_time_min       Tentative_identification  \
0         B01               1.979  Dihydroferulic acid 4-sulfate   
1         B02               2.048                    Quinic acid   
2         B03               2.529                   Pyrocatechol   
3         B04               2.943      Chlorogenic acid isomer I   
4         B05               3.271          4-Hydroxybenzoic acid   

  Molecular_formula  Monoisotopic_mass  Relative_abundance_EMB  \
0         C10H12O7S           276.0304                     3.5   
1           C7H12O6           192.0634                     6.3   
2            C6H6O2           110.0368                     NaN   
3          C16H18O9           354.0951                    10.1   
4            C7H6O3           138.0317                     1.5   

   Relative_abundance_EVB  
0                     NaN  
1                     7.4  
2                    13.6  
3                     0.5  
4                    18.5  

Tabla 4 (H

In [24]:
# Mapeo: Compound_ID final -> lista de IDs originales que representa
isomeros_fusionados = {
    "B04": ["B04", "B07", "B08"],  # chlorogenic acid isomers I, II, III
    "B06": ["B06", "B15"],          # coumaroylquinic acid isomers I, II
    "B10": ["B10", "B16"],          # feruloylquinic acid isomers I, II
}

def construir_texto_abundancia_pc(id_original, tabla):
    """Arma un texto tipo 'EMB=8.5%, EVB=4.4%' para un ID de la Tabla 3."""
    fila = tabla[tabla["Compound_ID"] == id_original]
    if fila.empty:
        return None
    emb = fila["Relative_abundance_EMB"].values[0]
    evb = fila["Relative_abundance_EVB"].values[0]
    emb_txt = f"{emb}%" if pd.notna(emb) else "no detectado"
    evb_txt = f"{evb}%" if pd.notna(evb) else "no detectado"
    return f"EMB={emb_txt}, EVB={evb_txt}"

def construir_texto_abundancia_hic(id_original, tabla):
    """Arma un texto tipo 'EMA=22.94%, EVA=7.60%' para un ID de la Tabla 4."""
    fila = tabla[tabla["Compound_ID"] == id_original]
    if fila.empty:
        return None
    ema = fila["Peak_area_EMA"].values[0]
    eva = fila["Peak_area_EVA"].values[0]
    ema_txt = f"{ema}%" if pd.notna(ema) else "no detectado"
    eva_txt = f"{eva}%" if pd.notna(eva) else "no detectado"
    return f"EMA={ema_txt}, EVA={eva_txt}"


for resultado in resultados_pubchem:
    compound_id = resultado["Compound_ID"]
    clase = resultado["Class"]

    # ¿Es un ID que representa varios isómeros fusionados?
    ids_a_consultar = isomeros_fusionados.get(compound_id, [compound_id])

    partes_abundancia = []
    for id_orig in ids_a_consultar:
        if clase == "PC":
            texto = construir_texto_abundancia_pc(id_orig, tabla3_original)
        else:
            texto = construir_texto_abundancia_hic(id_orig, tabla4_original)

        if texto is not None:
            partes_abundancia.append(f"{id_orig}: {texto}")

    resultado["Relative_Abundance"] = "; ".join(partes_abundancia) if partes_abundancia else None

print("✅ Columna Relative_Abundance construida para los 37 compuestos")
print("\nEjemplo (B04, isómeros fusionados):")
print(next(r["Relative_Abundance"] for r in resultados_pubchem if r["Compound_ID"] == "B04"))

✅ Columna Relative_Abundance construida para los 37 compuestos

Ejemplo (B04, isómeros fusionados):
B04: EMB=10.1%, EVB=0.5%; B07: EMB=2.8%, EVB=20.4%; B08: EMB=27.7%, EVB=4.3%


In [25]:
df_final = pd.DataFrame(resultados_pubchem)

columnas_finales = [
    "Compound_ID", "Name", "SMILES_Canonical", "InChIKey_RDKit",
    "Formula_Reported", "Extract", "Relative_Abundance", "Class"
]

df_final = df_final[columnas_finales].rename(columns={
    "SMILES_Canonical": "SMILES",
    "InChIKey_RDKit": "InChIKey",
    "Formula_Reported": "Formula",
})

ruta_salida = "../data/processed/step1_true_positive_compounds.csv"
df_final.to_csv(ruta_salida, index=False)

print(f"✅ Tabla final guardada en: {ruta_salida}")
print(f"Dimensiones: {df_final.shape[0]} filas x {df_final.shape[1]} columnas\n")

# Checklist actualizado, ahora sí incluyendo Relative_Abundance
columnas_requeridas = {"Compound_ID", "Name", "SMILES", "InChIKey", "Formula",
                       "Extract", "Relative_Abundance", "Class"}
faltantes = columnas_requeridas - set(df_final.columns)

print("="*60)
print("CHECKLIST DEL ENTREGABLE 1 (actualizado)")
print("="*60)
print(f"[{'✅' if len(df_final) >= 30 else '❌'}] Mínimo 30-35 compuestos → tienes {len(df_final)}")
print(f"[{'✅' if df_final['Compound_ID'].is_unique else '❌'}] Compound_ID sin duplicados")
print(f"[{'✅' if not faltantes else '❌'}] Todas las columnas requeridas presentes"
      f"{' (faltan: ' + str(faltantes) + ')' if faltantes else ''}")
print(f"[{'✅' if df_final['Relative_Abundance'].notna().all() else '⚠️'}] Relative_Abundance completo "
      f"({df_final['Relative_Abundance'].notna().sum()}/{len(df_final)})")

df_final

✅ Tabla final guardada en: ../data/processed/step1_true_positive_compounds.csv
Dimensiones: 37 filas x 8 columnas

CHECKLIST DEL ENTREGABLE 1 (actualizado)
[✅] Mínimo 30-35 compuestos → tienes 37
[✅] Compound_ID sin duplicados
[✅] Todas las columnas requeridas presentes
[✅] Relative_Abundance completo (37/37)


,Compound_ID,Name,SMILES,InChIKey,Formula,Extract,Relative_Abundance,Class
0,B01,Dihydroferulic acid 4-sulfate,COc1cc(CCC(=O)O)ccc1OS(=O)(=O)O,UMCDODPBPQMWQP-UHFFFAOYSA-N,C10H12O7S,EMB,"B01: EMB=3.5%, EVB=no detectado",PC
1,B02,Quinic acid,O=C(O)C1(O)CC(O)C(O)C(O)C1,AAWZDTNXLSGCEK-UHFFFAOYSA-N,C7H12O6,EMB;EVB,"B02: EMB=6.3%, EVB=7.4%",PC
2,B03,Pyrocatechol,Oc1ccccc1O,YCIMNLLNPGFGHC-UHFFFAOYSA-N,C6H6O2,EVB,"B03: EMB=no detectado, EVB=13.6%",PC
3,B04,Chlorogenic acid,O=C(C=Cc1ccc(O)c(O)c1)OC1CC(O)(C(=O)O)CC(O)C1O,CWVRJTMFETXNAD-UHFFFAOYSA-N,C16H18O9,EMB;EVB,"B04: EMB=10.1%, EVB=0.5%; B07: EMB=2.8%, EVB=2...",PC
4,B05,4-Hydroxybenzoic acid,O=C(O)c1ccc(O)cc1,FJKROLUGYXJWQN-UHFFFAOYSA-N,C7H6O3,EMB;EVB,"B05: EMB=1.5%, EVB=18.5%",PC
5,B06,p-Coumaroylquinic acid,O=C(C=Cc1ccc(O)cc1)OC1CC(O)(C(=O)O)CC(O)C1O,BMRSEYFENKXDIS-UHFFFAOYSA-N,C16H18O8,EMB;EVB,"B06: EMB=0.2%, EVB=21.5%; B15: EMB=1.3%, EVB=7.7%",PC
6,B09,Vanillin,COc1cc(C=O)ccc1O,MWOOGOJBHIARFG-UHFFFAOYSA-N,C8H8O3,EVB,"B09: EMB=no detectado, EVB=0.7%",PC
7,B10,Feruloylquinic acid,COc1cc(C=CC(=O)O[C@@H]2C[C@@](O)(C(=O)O)C[C@@H...,RAGZUCNPTLULOL-JSHWQEIDSA-N,C17H20O9,EMB;EVB,"B10: EMB=1.4%, EVB=32.5%; B16: EMB=10.6%, EVB=...",PC
8,B11,Vanillic acid,COc1cc(C(=O)O)ccc1O,WKOLLVMJNQIZCI-UHFFFAOYSA-N,C8H8O4,EVB,"B11: EMB=no detectado, EVB=2.4%",PC
9,B12,Caffeic acid,O=C(O)C=Cc1ccc(O)c(O)c1,QAIPRVGONGVQAS-UHFFFAOYSA-N,C9H8O4,EMB;EVB,"B12: EMB=10.4%, EVB=9.3%",PC


In [29]:
%pip install openpyxl
import openpyxl
from openpyxl.styles import Font, Alignment, PatternFill, Border, Side

wb = openpyxl.Workbook()
ws = wb.active
ws.title = "True-Positive Compounds"

# Escribimos encabezados
ws.append(list(df_final.columns))

# Escribimos cada fila de datos
for _, fila in df_final.iterrows():
    ws.append(list(fila))

# --- Estilos ---
font_name = "Arial"
header_fill = PatternFill(start_color="1F4E78", end_color="1F4E78", fill_type="solid")
header_font = Font(name=font_name, bold=True, color="FFFFFF", size=11)
normal_font = Font(name=font_name, size=10)
thin = Side(style="thin", color="CCCCCC")
border = Border(left=thin, right=thin, top=thin, bottom=thin)

for cell in ws[1]:
    cell.font = header_font
    cell.fill = header_fill
    cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
    cell.border = border

pc_fill = PatternFill(start_color="EAF1FA", end_color="EAF1FA", fill_type="solid")
hic_fill = PatternFill(start_color="FBF3E8", end_color="FBF3E8", fill_type="solid")
col_clase = list(df_final.columns).index("Class") + 1  # +1 porque openpyxl empieza en 1, no en 0

for r in range(2, ws.max_row + 1):
    clase = ws.cell(row=r, column=col_clase).value
    fill = pc_fill if clase == "PC" else hic_fill
    for c in range(1, ws.max_column + 1):
        cell = ws.cell(row=r, column=c)
        cell.font = normal_font
        cell.fill = fill
        cell.border = border
        cell.alignment = Alignment(vertical="center", wrap_text=True)

# Anchos de columna razonables
anchos = {"A": 12, "B": 34, "C": 55, "D": 24, "E": 13, "F": 14, "G": 40, "H": 8}
for col, w in anchos.items():
    ws.column_dimensions[col].width = w

ws.freeze_panes = "A2"
ws.row_dimensions[1].height = 30

ruta_excel = "../results/tables/step1_true_positive_compounds.xlsx"
wb.save(ruta_excel)
print(f"✅ Excel guardado en: {ruta_excel}")


[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


  Using cached openpyxl-3.1.5-py2.py3-none-any.whl.metadata (2.5 kB)
  Using cached et_xmlfile-2.0.0-py3-none-any.whl.metadata (2.7 kB)
Using cached openpyxl-3.1.5-py2.py3-none-any.whl (250 kB)
Using cached et_xmlfile-2.0.0-py3-none-any.whl (18 kB)
Note: you may need to restart the kernel to use updated packages.
✅ Excel guardado en: ../results/tables/step1_true_positive_compounds.xlsx


# 🔧 Corrección post-hoc: recuperación de estereoquímica perdida

*(Sección añadida al final del Paso 1, a partir de un hallazgo detectado durante
el Paso 2.0 — Mapeo de evidencia biológica)*

## Qué se descubrió

Al validar la búsqueda por InChIKey en ChEMBL para el compuesto B12 (Caffeic acid)
durante el Paso 2.0, se encontró que el InChIKey guardado en este Paso 1
(`QAIPRVGONGVQAS-UHFFFAOYSA-N`) **no coincide** con el InChIKey real y funcional
del compuesto en ChEMBL (`QAIPRVGONGVQAS-DUXPYHPUSA-N`) — a pesar de que ambos
comparten el mismo bloque de conectividad (`QAIPRVGONGVQAS`), difieren en la capa
de estereoquímica.

Se confirmó experimentalmente (generando ambos SMILES con y sin la configuración
*E* del doble enlace, y comparando el InChIKey resultante con RDKit) que el
InChIKey guardado corresponde a una versión **plana, sin estereoquímica** del
ácido cafeico — no a la molécula real, que tiene una configuración geométrica
específica.

## Causa raíz

La función de consulta a PubChem construida en este notebook pedía la propiedad
`ConnectivitySMILES` — que, por definición y diseño de PubChem, **excluye
intencionalmente** toda información de estereoquímica (nombre que ya deberíamos
haber interpretado como advertencia: "SMILES de solo conectividad"). El InChIKey
final de cada compuesto se recalculó con RDKit a partir de ese SMILES plano,
heredando la pérdida de estereoquímica — a pesar de que el propio campo `InChIKey`
que PubChem devuelve directamente en la misma respuesta (calculado internamente
por PubChem desde su registro completo) sí incluye la estereoquímica correcta.

Se confirmó además que la propiedad de PubChem que sí contiene la estereoquímica
(`IsomericSMILES`, tal como se solicita en la URL de consulta) se devuelve en la
práctica bajo la clave `SMILES` en la respuesta JSON — un tercer caso del mismo
patrón de renombrado silencioso de propiedades ya observado antes con
`CanonicalSMILES`/`ConnectivitySMILES`.

## Alcance del problema

De los 37 compuestos del true-positive set, se identificaron **9 candidatos**
con estereoquímica real conocida que probablemente se perdió con este método:
B02 (Quinic acid), B04 (Chlorogenic acid), B06 (p-Coumaroylquinic acid), B12
(Caffeic acid — confirmado), B18 (Schaftoside), B19 (Vicenin-2), B22 (Ferulic
acid), A18 (Levoglucosan), A26 (cis-Vaccenic acid). Los compuestos B10 y B21 ya
habían sido corregidos manualmente durante el Paso 1 original (con verificación
cruzada de fórmula e InChIKey completo contra PubChem) y no requieren corrección
aquí.

## Estrategia de corrección

En vez de corregir manualmente solo los 9 sospechosos (con riesgo de que algún
caso se nos escape), se vuelve a consultar **automáticamente los 35 compuestos**
resueltos originalmente por API (excluyendo únicamente B10 y B21), pidiendo esta
vez la estructura con estereoquímica y usando el InChIKey que PubChem calcula
directamente — no uno recalculado por RDKit a partir de una estructura
deliberadamente incompleta. Para los compuestos genuinamente sin estereocentros
(la mayoría), el resultado no debería cambiar; para los que sí tenían
estereoquímica real, se corrige.

## Validación

Cada corrección se verifica nuevamente por fórmula molecular (RDKit vs. paper
original), siguiendo el mismo estándar de control de calidad aplicado en todo
el Paso 1. El resultado se guarda primero en un archivo separado
(`step1_true_positive_compounds_CORREGIDO.csv`) para revisión antes de
reemplazar el archivo original usado como fuente en el Paso 2.

## Lección metodológica (tercera aparición del mismo patrón)

Ningún nombre de propiedad de una API pública debe asumirse sin verificación
directa de la respuesta cruda — patrón ya observado con `CanonicalSMILES`→
`ConnectivitySMILES` (Paso 1 original) y `confidence_score` para targets no
moleculares (Paso 2.0), y ahora con `IsomericSMILES` devuelto bajo la clave
`SMILES`.

In [8]:
##cor c

def obtener_estructura_correcta(nombre_busqueda):
    """
    Vuelve a consultar PubChem pidiendo explícitamente IsomericSMILES.
    HALLAZGO CONFIRMADO: PubChem devuelve esta propiedad bajo la clave 'SMILES'
    en el JSON, no bajo 'IsomericSMILES' — mismo patrón de renombrado silencioso
    que ya vimos con CanonicalSMILES/ConnectivitySMILES en el Paso 1 original.
    """
    nombre_codificado = requests.utils.quote(nombre_busqueda)
    url = (
        f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/{nombre_codificado}/"
        f"property/IsomericSMILES,InChIKey,MolecularFormula/JSON"
    )
    try:
        respuesta = requests.get(url, timeout=10)
        if respuesta.status_code != 200:
            return None
        datos = respuesta.json()
        propiedades = datos["PropertyTable"]["Properties"][0]
        return {
            "smiles_isomerico": propiedades.get("SMILES"),  # <- clave real confirmada arriba
            "inchikey_pubchem": propiedades.get("InChIKey"),
            "formula_pubchem": propiedades.get("MolecularFormula"),
        }
    except Exception as e:
        print(f"  ⚠️ Error con '{nombre_busqueda}': {e}")
        return None


# Confirmación con caffeic acid (ya sabemos qué esperar)
resultado = obtener_estructura_correcta("caffeic acid")
print(resultado)
print(f"¿Coincide con lo esperado? {resultado['inchikey_pubchem'] == 'QAIPRVGONGVQAS-DUXPYHPUSA-N'}")

{'smiles_isomerico': 'C1=CC(=C(C=C1/C=C/C(=O)O)O)O', 'inchikey_pubchem': 'QAIPRVGONGVQAS-DUXPYHPUSA-N', 'formula_pubchem': 'C9H8O4'}
¿Coincide con lo esperado? True


In [11]:
import pandas as pd

compuestos_37 = pd.read_csv("../data/processed/step1_true_positive_compounds.csv")
print(f"Compuestos cargados: {len(compuestos_37)}")
compuestos_37[["Compound_ID", "Name", "InChIKey"]]

Compuestos cargados: 37


,Compound_ID,Name,InChIKey
0,B01,Dihydroferulic acid 4-sulfate,UMCDODPBPQMWQP-UHFFFAOYSA-N
1,B02,Quinic acid,AAWZDTNXLSGCEK-UHFFFAOYSA-N
2,B03,Pyrocatechol,YCIMNLLNPGFGHC-UHFFFAOYSA-N
3,B04,Chlorogenic acid,CWVRJTMFETXNAD-UHFFFAOYSA-N
4,B05,4-Hydroxybenzoic acid,FJKROLUGYXJWQN-UHFFFAOYSA-N
5,B06,p-Coumaroylquinic acid,BMRSEYFENKXDIS-UHFFFAOYSA-N
6,B09,Vanillin,MWOOGOJBHIARFG-UHFFFAOYSA-N
7,B10,Feruloylquinic acid,RAGZUCNPTLULOL-JSHWQEIDSA-N
8,B11,Vanillic acid,WKOLLVMJNQIZCI-UHFFFAOYSA-N
9,B12,Caffeic acid,QAIPRVGONGVQAS-UHFFFAOYSA-N


In [12]:
#COR D
import time
from rdkit import Chem
from rdkit.Chem import rdMolDescriptors

IDS_YA_CORRECTOS = ["B10", "B21"]  # resueltos manualmente, con verificación cruzada completa (fórmula + InChIKey exacto contra ChEMBL)

cambios = []

for idx, row in compuestos_37.iterrows():
    compound_id = row["Compound_ID"]

    if compound_id in IDS_YA_CORRECTOS:
        print(f"⏭️  {compound_id}: ya verificado manualmente, se deja igual")
        continue

    nombre = row["Name"]
    resultado = obtener_estructura_correcta(nombre)
    time.sleep(0.3)

    if resultado is None or not resultado["smiles_isomerico"]:
        print(f"⚠️ {compound_id} ({nombre}): no se pudo re-consultar, se conserva el valor anterior")
        continue

    mol = Chem.MolFromSmiles(resultado["smiles_isomerico"])
    if mol is None:
        print(f"⚠️ {compound_id} ({nombre}): SMILES inválido según RDKit, se conserva el valor anterior")
        continue

    formula_rdkit = rdMolDescriptors.CalcMolFormula(mol)
    inchikey_rdkit = Chem.InchiToInchiKey(Chem.MolToInchi(mol))
    smiles_canonico = Chem.MolToSmiles(mol)

    inchikey_anterior = row["InChIKey"]
    cambio = inchikey_anterior != inchikey_rdkit

    if cambio:
        cambios.append({
            "Compound_ID": compound_id, "Name": nombre,
            "InChIKey_anterior": inchikey_anterior, "InChIKey_nuevo": inchikey_rdkit,
            "Formula_coincide": formula_rdkit == row["Formula"]
        })

    compuestos_37.at[idx, "SMILES"] = smiles_canonico
    compuestos_37.at[idx, "InChIKey"] = inchikey_rdkit

    estado = "🔄 CAMBIÓ" if cambio else "✅ igual (sin estereoquímica real)"
    print(f"{compound_id} ({nombre}): {estado}")

print(f"\n{'='*60}")
print(f"Total de compuestos con InChIKey corregido: {len(cambios)}")
pd.DataFrame(cambios)

B01 (Dihydroferulic acid 4-sulfate): ✅ igual (sin estereoquímica real)
B02 (Quinic acid): 🔄 CAMBIÓ
B03 (Pyrocatechol): ✅ igual (sin estereoquímica real)
B04 (Chlorogenic acid): 🔄 CAMBIÓ
B05 (4-Hydroxybenzoic acid): ✅ igual (sin estereoquímica real)
B06 (p-Coumaroylquinic acid): 🔄 CAMBIÓ
B09 (Vanillin): ✅ igual (sin estereoquímica real)
⏭️  B10: ya verificado manualmente, se deja igual
B11 (Vanillic acid): ✅ igual (sin estereoquímica real)
B12 (Caffeic acid): 🔄 CAMBIÓ
B13 (4-Hydroxybenzaldehyde): ✅ igual (sin estereoquímica real)
B14 (2,4-Dihydroxybenzoic acid): ✅ igual (sin estereoquímica real)
B17 (p-Coumaric acid): 🔄 CAMBIÓ
B18 (Schaftoside): 🔄 CAMBIÓ
B19 (Vicenin-2): 🔄 CAMBIÓ
B20 (3,4-Dihydroxybenzaldehyde): ✅ igual (sin estereoquímica real)
⏭️  B21: ya verificado manualmente, se deja igual
B22 (Ferulic acid): 🔄 CAMBIÓ


[17:13:45] WARNING: Omitted undefined stereo



A03 (4-Hydroxy-2,5-dimethyl-3(2H)-furanone): ✅ igual (sin estereoquímica real)


[17:13:47] WARNING: Omitted undefined stereo



A05 (3-Methylcyclopentane-1,2,4-trione): ✅ igual (sin estereoquímica real)
A06 (Glycerol): ✅ igual (sin estereoquímica real)


[17:13:49] WARNING: Omitted undefined stereo



A07 (2,3-Dihydro-3,5-dihydroxy-6-methyl-4H-pyran-4-one): ✅ igual (sin estereoquímica real)
A09 (2-Methoxy-4-vinylphenol): ✅ igual (sin estereoquímica real)
A10 (4-Methylcatechol): ✅ igual (sin estereoquímica real)
A13 (2,6-Dimethoxyphenol): ✅ igual (sin estereoquímica real)
A15 (Pyrogallol): ✅ igual (sin estereoquímica real)
A17 (2,4-Di-tert-butylphenol): ✅ igual (sin estereoquímica real)
A18 (Levoglucosan): 🔄 CAMBIÓ
A19 (4-Butyl-2-methoxyphenol): ✅ igual (sin estereoquímica real)
A22 (1-Hydroxycyclohexyl phenyl ketone): ✅ igual (sin estereoquímica real)
A23 (Myristic acid): ✅ igual (sin estereoquímica real)
A24 (Palmitic acid): ✅ igual (sin estereoquímica real)
A25 (3-(3,5-Di-tert-butyl-4-hydroxyphenyl)propionic acid): ✅ igual (sin estereoquímica real)
A26 (Vaccenic acid): 🔄 CAMBIÓ
A27 (Stearic acid): ✅ igual (sin estereoquímica real)
A28 (Arachidic acid): ✅ igual (sin estereoquímica real)
A29 (3,4-Dimethoxyphenol): ✅ igual (sin estereoquímica real)

Total de compuestos con InChIKey c

,Compound_ID,Name,InChIKey_anterior,InChIKey_nuevo,Formula_coincide
0,B02,Quinic acid,AAWZDTNXLSGCEK-UHFFFAOYSA-N,AAWZDTNXLSGCEK-LNVDRNJUSA-N,True
1,B04,Chlorogenic acid,CWVRJTMFETXNAD-UHFFFAOYSA-N,CWVRJTMFETXNAD-JUHZACGLSA-N,True
2,B06,p-Coumaroylquinic acid,BMRSEYFENKXDIS-UHFFFAOYSA-N,BMRSEYFENKXDIS-LUTKEZBSSA-N,True
3,B12,Caffeic acid,QAIPRVGONGVQAS-UHFFFAOYSA-N,QAIPRVGONGVQAS-DUXPYHPUSA-N,True
4,B17,p-Coumaric acid,NGSWKAQJJWESNS-UHFFFAOYSA-N,NGSWKAQJJWESNS-ZZXKWVIFSA-N,True
5,B18,Schaftoside,MMDUKUSNQNWVET-UHFFFAOYSA-N,MMDUKUSNQNWVET-VYUBKLCTSA-N,True
6,B19,Vicenin-2,FIAAVMJLAGNUKW-UHFFFAOYSA-N,FIAAVMJLAGNUKW-VQVVXJKKSA-N,True
7,B22,Ferulic acid,KSEBMYQBYZTDHS-UHFFFAOYSA-N,KSEBMYQBYZTDHS-HWKANZROSA-N,True
8,A18,Levoglucosan,TWNIBLMWSKIRAT-UHFFFAOYSA-N,TWNIBLMWSKIRAT-VFUOTHLCSA-N,True
9,A26,Vaccenic acid,UWHZIFQPPBDJPM-UHFFFAOYSA-N,UWHZIFQPPBDJPM-BQYQJAHWSA-N,True


In [13]:
#COR E
df_cambios = pd.DataFrame(cambios)
print(f"Total de cambios: {len(df_cambios)}")
print(f"¿Todas las fórmulas siguen coincidiendo? {df_cambios['Formula_coincide'].all()}")
df_cambios


Total de cambios: 10
¿Todas las fórmulas siguen coincidiendo? True


,Compound_ID,Name,InChIKey_anterior,InChIKey_nuevo,Formula_coincide
0,B02,Quinic acid,AAWZDTNXLSGCEK-UHFFFAOYSA-N,AAWZDTNXLSGCEK-LNVDRNJUSA-N,True
1,B04,Chlorogenic acid,CWVRJTMFETXNAD-UHFFFAOYSA-N,CWVRJTMFETXNAD-JUHZACGLSA-N,True
2,B06,p-Coumaroylquinic acid,BMRSEYFENKXDIS-UHFFFAOYSA-N,BMRSEYFENKXDIS-LUTKEZBSSA-N,True
3,B12,Caffeic acid,QAIPRVGONGVQAS-UHFFFAOYSA-N,QAIPRVGONGVQAS-DUXPYHPUSA-N,True
4,B17,p-Coumaric acid,NGSWKAQJJWESNS-UHFFFAOYSA-N,NGSWKAQJJWESNS-ZZXKWVIFSA-N,True
5,B18,Schaftoside,MMDUKUSNQNWVET-UHFFFAOYSA-N,MMDUKUSNQNWVET-VYUBKLCTSA-N,True
6,B19,Vicenin-2,FIAAVMJLAGNUKW-UHFFFAOYSA-N,FIAAVMJLAGNUKW-VQVVXJKKSA-N,True
7,B22,Ferulic acid,KSEBMYQBYZTDHS-UHFFFAOYSA-N,KSEBMYQBYZTDHS-HWKANZROSA-N,True
8,A18,Levoglucosan,TWNIBLMWSKIRAT-UHFFFAOYSA-N,TWNIBLMWSKIRAT-VFUOTHLCSA-N,True
9,A26,Vaccenic acid,UWHZIFQPPBDJPM-UHFFFAOYSA-N,UWHZIFQPPBDJPM-BQYQJAHWSA-N,True


In [ ]:
COR#F
ruta_corregida = "../data/processed/step1_true_positive_compounds_CORREGIDO.csv"
compuestos_37.to_csv(ruta_corregida, index=False)
print(f"✅ Guardado en: {ruta_corregida}")

✅ Guardado en: ../data/processed/step1_true_positive_compounds_CORREGIDO.csv
